# Milestone - Unit 7: semantic text embeddings -- a dense content path that matches books by the *meaning* of their words (pretrained GloVe + brute-force cosine), honestly weaker than lexical here yet worth adding because it surfaces different books and lifts both the blend's hit *and* its coverage.

## 1. Load the catalog, interactions, keyword documents, cold readers, and the committed GloVe subset

The same generated substrate Units 1-6 ran on, plus the **committed GloVe subset** (`recsys/data/glove/glove_subset.{npy,json}`): a checksum-pinned, vocabulary-restricted slice of `glove-wiki-gigaword-100` (PDDL-1.0 vector data), loaded with **numpy only** -- no gensim. It covers 607 of the catalog's 608 keyword tokens (only `starfall` is out of vocabulary) in 100 dimensions.
`bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    book_embedding,
    catalog_coverage,
    hit_rate_at_k,
    load_glove_subset,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)
from bookrec.lexical import tokenize

K = 10
POOL = 30
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

glove = load_glove_subset()

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "cold_readers": len(cold_readers),
    "glove_vocab": len(glove.tokens),
    "glove_dim": glove.dim,
    "glove_out_of_vocab": glove.missing,
}

## 2. Fit the five Part-1 paths and the new semantic path

Six signals, each a `RetrievalPath` with the same `fit`/`retrieve` contract: the seeded random floor, crowd popularity, Unit-3 **lexical** content words (sparse bag-of-words), Unit-4 item-item CF, Unit-5 matrix factorization (the one ~15 s training fit, at its pinned config), and the new Unit-7 **semantic** path. Like `LexicalRetrievalPath`, the semantic path takes the keyword corpus *and* the GloVe subset at construction, so `fit` stays protocol-substitutable and merely precomputes each book's L2-normalized embedding (content only -- leakage-safe). Every seed is fixed, so the demo is deterministic.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=RANDOM_SEED).fit(rows, catalog=catalog_ids)  # ~15 s
semantic = bookrec.SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf, mf, semantic):
    registry.register(path)

[path.name for path in registry]

## 3. Peek inside the semantic space: nearest words

A GloVe vector turns each word into a point in a 100-dimensional space where **cosine similarity reflects meaning**. Below are the nearest words to a few query tokens -- but read the label carefully: these are the **nearest words among this catalog's 607-word vocabulary**, not full-GloVe analogies (the committed subset only holds the slice vocabulary). Even so, `dragon` lands near `demon`/`sword`/`monster`, `robot` near `spaceship`/`alien`/`cyborg`, and `ocean` near `shore`/`island`/`reef`: words that never *are* each other still sit close when they *mean* something related.

In [ ]:
# L2-normalize every GloVe row so a dot product IS the cosine. The one out-of-vocabulary token
# (`starfall`) is a zero row; guard its zero norm so the division stays finite.
norms = np.linalg.norm(glove.matrix, axis=1, keepdims=True)
norms[norms == 0.0] = 1.0
unit_matrix = glove.matrix / norms

def nearest_words(word, n=5):
    """The n nearest words to `word` AMONG THE CATALOG'S 607-WORD VOCABULARY (not full GloVe)."""
    row = glove.token_index.get(word)
    if row is None:
        return []
    sims = unit_matrix @ unit_matrix[row]
    order = np.argsort(-sims)
    return [(glove.tokens[i], round(float(sims[i]), 3)) for i in order if glove.tokens[i] != word][:n]

{word: nearest_words(word) for word in ("dragon", "robot", "ocean", "detective")}

## 4. The `val` scoreboard -- semantic beside random, popularity, lexical, CF, and MF

A book embedding is the **mean of its keyword tokens' GloVe vectors, L2-normalized** (out-of-vocabulary tokens skipped); a reader's query embedding is the mean of the books in their `seen` set; retrieval scores every catalog book by cosine and drops the seen ones. We score all six paths on the frozen `val` split with `run_validation_scoreboard` (the same 500 eligible readers, the 60 cold readers excluded), add catalog coverage from a shared harness, and **assert the harness hit matches the official scoreboard** for every deterministic path -- binding these numbers to shipped code.

In [ ]:
known_items = set(catalog_ids)
cold_set = set(cold_readers)

seen_by_reader = defaultdict(set)
relevant_by_reader = defaultdict(set)
for row in rows:
    if row["item_id"] not in known_items or int(row["label"]) != 1:
        continue
    if row["split"] == "train":
        seen_by_reader[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        relevant_by_reader[row["reader_id"]].add(row["item_id"])

eligible_readers = [
    reader_id
    for reader_id in sorted(relevant_by_reader)
    if reader_id not in cold_set and (relevant_by_reader[reader_id] - seen_by_reader[reader_id])
]

single_rows = []
for path in registry:
    board = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    rec_lists = [
        rank(path.retrieve(rid, {"seen": seen_by_reader[rid]}, K), n=K, exclude=seen_by_reader[rid])
        for rid in eligible_readers
    ]
    coverage = catalog_coverage(rec_lists, catalog_ids)
    harness_hit = sum(
        hit_rate_at_k(recs, relevant_by_reader[rid] - seen_by_reader[rid], K)
        for recs, rid in zip(rec_lists, eligible_readers)
    ) / len(eligible_readers)
    # Deterministic paths reproduce the official scoreboard exactly; the random floor is stateful
    # (its RNG has already advanced inside the scoreboard), so it is exempt from the equality check.
    if path.name != "random":
        assert abs(harness_hit - board.hit_rate_at_k) < 1e-9, path.name
    single_rows.append({
        "path": path.name,
        "hit@10": round(board.hit_rate_at_k, 4),
        "recall@10": round(board.recall_at_k, 4),
        "precision@10": round(board.precision_at_k, 4),
        "ndcg@10": round(board.ndcg_at_k, 4),
        "coverage": round(coverage, 4),
    })

scoreboard = {entry["path"]: entry for entry in single_rows}

# The ground-truth numbers the lesson quotes (seed 0, k=10, 500 val readers).
assert abs(scoreboard["random"]["hit@10"] - 0.012) < 0.001
assert abs(scoreboard["popularity"]["hit@10"] - 0.108) < 0.001
assert abs(scoreboard["lexical"]["hit@10"] - 0.158) < 0.001
assert abs(scoreboard["item-item"]["hit@10"] - 0.252) < 0.001
assert abs(scoreboard["mf"]["hit@10"] - 0.276) < 0.001
assert abs(scoreboard["semantic"]["hit@10"] - 0.102) < 0.001
assert abs(scoreboard["semantic"]["recall@10"] - 0.051) < 0.001
assert abs(scoreboard["semantic"]["ndcg@10"] - 0.0325) < 0.001
# Semantic clears the random floor ~8.5x but sits BELOW lexical (~0.65x) -- the honest result.
assert scoreboard["semantic"]["hit@10"] > 8 * scoreboard["random"]["hit@10"]
assert scoreboard["semantic"]["hit@10"] < scoreboard["lexical"]["hit@10"]

pd.DataFrame(single_rows)

Read it honestly. Semantic lands at **hit@10 ~0.102** -- about **8.5x the random floor** (0.012), so the meaning signal is real -- but **below lexical's 0.158** (~0.65x), and far below the collaborative CF (0.252) and MF (0.276). **Why is it below lexical here?** Our synthetic catalog's 32 latent-taste topics are *arbitrary contiguous slices* of a word pool, so GloVe's meaning-geometry is **uncorrelated-by-construction** with the latent taste structure; only the 12 genre word-banks carry GloVe-exploitable meaning. On a real catalog's prose, dense-semantic matching would fare differently -- we teach the *method* honestly, not a staged win. **Do not read this as 'embeddings are worse'**: it is a synthetic-corpus artifact. What makes semantic worth keeping is not raw accuracy -- it is **complementarity**, which the next three sections measure.

## 5. One reader's semantic recommendations

For a single reader we build one book embedding by hand -- the mean of a seen book's keyword tokens' GloVe vectors, L2-normalized -- and confirm its norm is 1. Then we read the reader's top semantic recommendations: the titles are generated placeholders, so the **keyword bags** tell the story. The recommended books share *meaning* (fantasy/nature/creature words) with the reader's seen books even when the exact surface tokens differ -- that is what a dense embedding buys over sparse word-matching.

In [ ]:
reader_id = 3
seen = seen_by_reader[reader_id]
reader_recs = rank(semantic.retrieve(reader_id, {"seen": seen}, 5), n=5, exclude=seen)

# Build one book embedding by hand; it is unit-norm (mean of token vectors, then L2-normalized).
sample_book = min(seen)
sample_embedding = book_embedding(tokenize(keywords[sample_book]), glove)

{
    "reader": reader_id,
    "seen_keyword_bags": {book: tokenize(keywords[book])[:6] for book in sorted(seen)[:3]},
    "hand_built_embedding_norm": round(float(np.linalg.norm(sample_embedding)), 3),
    "recommended": [
        {
            "book": candidate.item_id,
            "cosine": round(candidate.score, 3),
            "keywords": tokenize(keywords[candidate.item_id])[:6],
        }
        for candidate in reader_recs
    ],
}

## 6. Complementarity (1): the lexical-vs-semantic top-10 overlap

If semantic just re-found the same books as lexical, it would add nothing. So we measure, for every eligible reader, the fraction of the top 10 the two paths **share**, and average it. A low overlap means the two paths surface *different* books -- complementary, not redundant.

In [ ]:
overlap_fractions = []
for rid in eligible_readers:
    seen = seen_by_reader[rid]
    semantic_top = {c.item_id for c in semantic.retrieve(rid, {"seen": seen}, K)}
    lexical_top = {c.item_id for c in lexical.retrieve(rid, {"seen": seen}, K)}
    if semantic_top and lexical_top:
        overlap_fractions.append(len(semantic_top & lexical_top) / K)

lexical_semantic_overlap = sum(overlap_fractions) / len(overlap_fractions)
# Measured ~0.216: the two paths largely surface DIFFERENT books.
assert abs(lexical_semantic_overlap - 0.216) < 0.005

{
    "readers_compared": len(overlap_fractions),
    "mean_top10_overlap": round(lexical_semantic_overlap, 4),
}

Only **~0.216** of the top 10 is shared -- barely two books in ten. GloVe relates books by *word meaning* where lexical BM25 needs *shared surface tokens*, so semantic surfaces candidates lexical misses. That is exactly the raw material a blend turns into a measurable gain.

## 7. Complementarity (2): the ablation -- the blend WITH vs WITHOUT the semantic path

The quantitative complementarity test. We take the **Unit-6 pinned blend** -- weights `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}` at per-path pool 30, exactly the blend Unit 6 shipped and Checkpoint A froze -- then add the semantic path (weight 0.5) and re-measure hit@10 and catalog coverage with `run_blended_scoreboard`. If semantic truly complements the others, adding it -- *despite being the weakest single path* -- should lift the blend.

In [ ]:
PINNED_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
blend_paths = {"popularity": popularity, "lexical": lexical, "item-item": cf, "mf": mf}

board_without = run_blended_scoreboard(
    blend_paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights=PINNED_WEIGHTS, cold_readers=cold_readers,
)
board_with = run_blended_scoreboard(
    {**blend_paths, "semantic": semantic}, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights={**PINNED_WEIGHTS, "semantic": 0.5}, cold_readers=cold_readers,
)

# Measured (Unit-6 pinned blend, pool 30): hit 0.306 -> 0.316 AND coverage 0.333 -> 0.353.
# Adding the weakest single path lifts BOTH.
assert abs(board_without.hit_rate_at_k - 0.306) < 0.003
assert abs(board_without.catalog_coverage - 0.333) < 0.003
assert abs(board_with.hit_rate_at_k - 0.316) < 0.003
assert abs(board_with.catalog_coverage - 0.353) < 0.003
assert board_with.hit_rate_at_k > board_without.hit_rate_at_k
assert board_with.catalog_coverage > board_without.catalog_coverage

print(f"blend WITHOUT semantic: hit@10={board_without.hit_rate_at_k:.4f} coverage={board_without.catalog_coverage:.4f}")
print(f"blend WITH    semantic: hit@10={board_with.hit_rate_at_k:.4f} coverage={board_with.catalog_coverage:.4f}")

**The honest headline.** Semantic is the weakest single path on `val` (0.102, below even lexical), yet adding it to the blend lifts **both** hit@10 (0.306 -> 0.316) **and** coverage (0.333 -> 0.353). That is complementarity made quantitative: a path need not win on its own to earn its place -- it needs to surface books the others miss (overlap only ~0.216), and this one does.

**The bridge.** Semantic retrieval scores a reader embedding against book embeddings by a dot product / cosine -- the *same* retriever that scores Unit 5's MF factors. The only difference is where the vectors come from: GloVe's are **pretrained** on text, MF's are **learned** from the interaction log. Unit 8's two-tower will make them **learned *and* neural** -- but still a dot product over embeddings. One retrieval idea, three kinds of vector.